## Gold Layer

Building gold layer based on dimensional model. Creating the fct_results, dim_event, dim_athelete, and views for each marathon typ.

In [0]:
from pyspark.sql.functions import col
# reading in from silver OBT
df = spark.read.table("marathos.silver.races_obt")
print(f"Rows loaded from the silver:{df.count()}")

## Fact Table - fct_results
Here i am creating the central fact table with the performance metrics and foregin keys to the dimensions. 

In [0]:
# Creating fct_results - central fact table 

fct_results = df.select(
    col("result_id"),
    col("event_id"),
    col("athlete_id"),
    col("year_of_event"),
    col("athlete_performance"),
    col("performance_seconds"),
    col("performance_km"),
    col("athlete_average_speed")
)
fct_results.write.format("delta").saveAsTable("marathos.gold.fct_results")
print(f"fct_results written: {fct_results.count()} rows")

## Dimension Table - dim_event
Creating the event dimension with unique events and attributes

In [0]:
# CReating dim_event - Unique events only 

dim_event = df.select(
    col("event_id"),
    col("event_name"),
    col("event_distance_length"),
    col("event_number_of_finishers"),
    col("event_dates"),
    col("year_of_event")
).dropDuplicates(["event_id"])

dim_event.write.format("delta").saveAsTable("marathos.gold.dim_event")
print(f"dim_event written: {dim_event.count()} rows")


## Dimension Table - dim_athlete
creating athele dimension with unique athletes only 

In [0]:
# Create dim_athlete - unique athletes only
dim_athlete = df.select(
    col("athlete_id"),
    col("athlete_country"),
    col("athlete_year_of_birth"),
    col("athlete_gender"),
    col("athlete_age_category"),
    col("athlete_club")
).dropDuplicates(["athlete_id"])

dim_athlete.write.format("delta").saveAsTable("marathos.gold.dim_athlete")
print(f"dim_athlete written: {dim_athlete.count()} rows")

## Views per Marathon Type
Creating at least two views for each marathon type as required by stakeholder,
Distance events: races measured in km or mi where performance is in time and should be = (h)
Time events: races is measured in h where the performance is in distance (km)

In [0]:
## View 1 distance events - top "performers" by the speed 
spark.sql("""
    CREATE OR REPLACE VIEW marathos.gold.vw_distance_top_performers AS
    SELECT
        f.result_id,
        f.athlete_id,
        f.athlete_average_speed,
        f.performance_seconds,
        f.year_of_event,
        e.event_name,
        e.event_distance_length,
        a.athlete_country,
        a.athlete_gender
    FROM marathos.gold.fct_results f
    JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
    JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
    WHERE f.performance_seconds IS NOT NULL
    ORDER BY f.athlete_average_speed DESC

""")
print("vw_distance_top_performers created")

## View 2: Distance events - results per country 

In [0]:
## View 2  distance events - participants per country
spark.sql("""
    CREATE OR REPLACE VIEW marathos.gold.vw_distance_by_country AS
    SELECT
        a.athlete_country,
        COUNT(*) as total_results,
        AVG(f.athlete_average_speed) as avg_speed,
        AVG(f.performance_seconds) as avg_performance_seconds,
        f.year_of_event
    FROM marathos.gold.fct_results f
    JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
    JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
    WHERE f.performance_seconds IS NOT NULL
    GROUP BY a.athlete_country, f.year_of_event
    ORDER BY total_results DESC
""")

print("vw_distance_by_country created")

## View 3: Time events - top performers by distance 

In [0]:
## View 3: Time events - Top performers by distance covered . 

spark.sql("""
    CREATE OR REPLACE VIEW marathos.gold.vw_time_top_performers AS   
    SELECT
        f.result_id,
        f.athlete_id,
        f.performance_km,
        f.athlete_average_speed,
        f.year_of_event,
        e.event_name,
        e.event_distance_length,
        a.athlete_country,
        a.athlete_gender
    FROM marathos.gold.fct_results f 
    JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
    JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
    WHERE f.performance_km IS NOT NULL
    ORDER BY f.performance_km DESC
""")
print("vw_time_top_performers created")

## Time events: This is for the participants per country 

In [0]:
## View 4 Time events - for participants per country

spark.sql("""
    CREATE OR REPLACE VIEW marathos.gold.vw_time_by_country AS
    SELECT
        a.athlete_country,
        COUNT(*) as total_results,
        AVG(f.performance_km) as avg_distance_km,
        f.year_of_event
    FROM marathos.gold.fct_results f 
    JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
    JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
    WHERE f.performance_km IS NOT NULL
    GROUP BY a.athlete_country, f.year_of_event
    ORDER BY total_results DESC    
""")

print("vw_time_by_country created") 

## Verification
Now it is important that i all the gold tables and views where created succsessfully.

In [0]:
# Verifying all tables and views exist
print("Gold layer verification")
print(f"fct_results: {spark.read.table('marathos.gold.fct_results').count()} rows")
print(f"dim_event: {spark.read.table('marathos.gold.dim_event').count()} rows")
print(f"dim_athlete: {spark.read.table('marathos.gold.dim_athlete').count()} rows")

spark.sql("SELECT COUNT(*) as rows FROM marathos.gold.vw_distance_top_performers").show()
spark.sql("SELECT COUNT(*) as rows FROM marathos.gold.vw_distance_by_country").show()
spark.sql("SELECT COUNT(*) as rows FROM marathos.gold.vw_time_top_performers").show()
spark.sql("SELECT COUNT(*) as rows FROM marathos.gold.vw_time_by_country").show()